In [1]:
import hist
from coffea.util import load, save
import mplhep as hep
import matplotlib.pyplot as plt
import copy
import numpy as np
import os
from tabulate import tabulate
import uncertainties as unc  
import uncertainties.unumpy as unumpy 
plt.style.use(hep.style.CMS)


In [149]:
year = "2022post"
histversion = year #'2022pre'#22Test'#'GenIsoPhoTest' # 'sdfjTest'#
whathist = {
    '2022pre': '2022_0616_test1',
    '2022post':'2022EE_0616',
    '2023pre': '2023_0616',
    '2023post': '2023BPix_0529'

}
lumi = {
    '2018': 59.83,
    '2022pre' : 7.99,
    '2022post': 26.68,
    '2023pre': 17.96,
    '2023post': 9.68
}
if 'Test' in histversion:
    hists = load('./hadmonotop'+whathist[histversion]+'.scaled')
else:
    hists = load('./hadmonotop'+whathist[histversion]+'.scaled')
    #hists = load('./hadmonotop'+year+'_'+whathist[histversion]+'.scaled')

hists_bkg = hists['bkg']
hists_data= hists['data']


## Yields

In [128]:
############
## Yields
############
TPhi = '' #'TPhiTo2Chi_MPhi200_MChi150_TuneCP5_13TeV-amcatnlo-pythia8'
variables = list(hists_bkg.keys())
process = list(hists_bkg['sumw'].keys())
print(process)
region = 'wmcr'
regions = ['sr','tecr', 'tmcr', 'wecr', 'wmcr', 'zecr', 'zmcr', 'gcr']
print(f'** Yields table for {year} {lumi[year]} fb-1 **')
v = 'ut'
for reg in regions:

    if 's' in reg:
        data_name = 'MET'
    elif 'e' in reg:
        data_name = 'EGamma'
        #continue
    elif 'm' in reg:
        data_name = 'MET'
    else:
        data_name = 'EGamma'
        #continue
    bkg_all = 0
    print('======== ', reg, ' ========')
    for proc in process:
        #if 'Z' in proc : continue
        if reg not in list(hists_bkg[v][proc].axes[0]) :
            print("%-25s" % proc, "%10.2f" % 0.0)
        else:
            print("%-25s" % proc, "%10.2f" % np.sum(hists_bkg[v][proc][{'region': reg}].values()))
            bkg_all = bkg_all + np.sum(hists_bkg[v][proc][{'region': reg}].values())
    print('------------------------------------')
    print("%-25s" % str('BKG '), "%10.2f" % bkg_all)
    print("%-25s" % str('DATA'), "%10.2f" % np.sum(hists_data[v][data_name][{'region': reg}].values()))
    print()

['WW', 'WZ', 'ZZ', 'QCD Multijet', 'TT', 'ST', 'Z ($\\nu\\nu$) + Jets', 'Z ($\\ell\\ell$) + Jets', 'W ($\\ell\\nu$) + Jets', 'G + Jets']
** Yields table for 2022post 26.68 fb-1 **
========  sr  ========
WW                            212.48
WZ                            338.61
ZZ                            246.81
QCD Multijet                  424.39
TT                           1068.08
ST                            243.09
Z ($\nu\nu$) + Jets         49172.92
Z ($\ell\ell$) + Jets         218.83
W ($\ell\nu$) + Jets        23569.77
G + Jets                     1647.71
------------------------------------
BKG                         77142.68
DATA                        65899.00

========  tecr  ========
WW                             41.85
WZ                             12.98
ZZ                              0.44
QCD Multijet                    8.42
TT                           2305.94
ST                            519.13
Z ($\nu\nu$) + Jets             0.14
Z ($\ell\ell$) + Jets          

## DATA cutflow

In [122]:
regions = ['tecr', 'tmcr', 'wecr', 'wmcr', 'zecr', 'zmcr', 'gcr']
mcset = 'EGamma'
data_cutflow = hists_data['cutflow']
print(f'** DATA cutflow for {year} {lumi[year]} fb-1 **')
for reg in regions:
    if 'm' in reg:
        #continue
        dataset = 'MET'
    else:
        dataset = 'EGamma'
    
    cutlists=data_cutflow[dataset].axes[1]
    yd = hists['data']['cutflow'][dataset][{'region':reg}].values()[()]
    real_yd = np.sum(yd, axis=1)
    show_me = {}
    for idx, cut in enumerate(cutlists):
        if real_yd[idx] == 0.0:
            #print('value zero %s' % cut)
            continue
        else:
            show_me[cut] = real_yd[idx]


    real_cutflow = {k: v for k, v in sorted(show_me.items(), key=lambda item: item[1], reverse=True)}

    print()
    #print("== Hist version is ", whathist[histversion], " ==")
    print("          == [", reg, "] ==")
    for key in real_cutflow:
        if 'exclude' in key: continue
        print("%-27s" % key, "%13.2f" % real_cutflow[key])
    print()

** DATA cutflow for 2022post 26.68 fb-1 **

          == [ tecr ] ==
Initial                      657164164.00
lumimask                     657164164.00
met_filters                  654689719.00
single_electron_triggers     289108431.00
QCD_NoGenIsoPho              289108431.00
recoil_tecr                      82841.00
mindphi_tecr                     65202.00
GJet_GenIsoPho                   61965.00
minDphi_tecr                     61965.00
jetveto                          58366.00
one_ak15                         58366.00
leading_fj250                    57429.00
isoneE                           50110.00
met150                           27772.00
extrab                            4760.00


          == [ tmcr ] ==
Initial                      707641071.00
lumimask                     707641071.00
met_filters                  692880896.00
met_triggers                  44967376.00
QCD_NoGenIsoPho               44967376.00
recoil_tmcr                      89906.00
mindphi_tmcr          

## Background cutflow

In [150]:
regions = ['sr', 'tecr', 'tmcr', 'wecr', 'wmcr', 'zecr', 'zmcr', 'gcr']
dataset = f'Z ($\\nu\\nu$) + Jets'
data_cutflow = hists_bkg['cutflow']
print(f'** {dataset} cutflow for {year} {lumi[year]} fb-1 **')
for reg in regions:
    
    cutlists=data_cutflow[dataset].axes[1]
    yd = hists['bkg']['cutflow'][dataset][{'region':reg}].values()[()]
    real_yd = np.sum(yd, axis=1)
    show_me = {}
    for idx, cut in enumerate(cutlists):
        if real_yd[idx] == 0.0:
            #print('value zero %s' % cut)
            continue
        else:
            show_me[cut] = real_yd[idx]


    real_cutflow = {k: v for k, v in sorted(show_me.items(), key=lambda item: item[1], reverse=True)}

    print()
    #print("== Hist version is ", whathist[histversion], " ==")
    print("          == [", reg, "] ==")
    for key in real_cutflow:
        if 'exclude' in key: continue
        print("%-27s" % key, "%13.2f" % real_cutflow[key])
    print()

** Z ($\nu\nu$) + Jets cutflow for 2022post 26.68 fb-1 **

          == [ sr ] ==
Initial                               inf
lumimask                              inf
met_filters                           inf
met_triggers                          nan
QCD_NoGenIsoPho                       nan
recoil_sr                             nan
mindphi_sr                            nan
minDphi_sr                            nan
GJet_GenIsoPho                        nan
jetveto                               nan
one_ak15                              nan
leading_fj250                         nan
iszeroL                               nan
noextrab                              nan


          == [ tecr ] ==
Initial                               inf
lumimask                              inf
met_filters                           inf
QCD_NoGenIsoPho                       nan
GJet_GenIsoPho                        nan
jetveto                               nan
one_ak15                              nan
leading_f

In [151]:
hists_bkg['cutflow'].keys()

dict_keys(['WW', 'WZ', 'ZZ', 'QCD Multijet', 'TT', 'ST', 'Z ($\\nu\\nu$) + Jets', 'Z ($\\ell\\ell$) + Jets', 'W ($\\ell\\nu$) + Jets', 'G + Jets'])

In [152]:
np.sum(hists_bkg['cutflow']['Z ($\\nu\\nu$) + Jets'][{'region':'sr'}].values()[()], axis=1)

array([inf, inf, inf, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.])

In [153]:
hists_bkg['cutflow']['Z ($\\nu\\nu$) + Jets'][{'region':'sr'}].values()[()]

array([[             inf,       0.        ,       0.        ,
              0.        ,       0.        ,       0.        ,
              0.        ,       0.        ,       0.        ,
              0.        ,       0.        ,       0.        ,
              0.        ,       0.        ,       0.        ,
              0.        ,       0.        ],
       [      0.        ,              inf,       0.        ,
              0.        ,       0.        ,       0.        ,
              0.        ,       0.        ,       0.        ,
              0.        ,       0.        ,       0.        ,
              0.        ,       0.        ,       0.        ,
              0.        ,       0.        ],
       [      0.        ,       0.        ,              inf,
              0.        ,       0.        ,       0.        ,
              0.        ,       0.        ,       0.        ,
              0.        ,       0.        ,       0.        ,
              0.        ,       0.        

In [133]:
cutlists

StrCategory(['Initial', 'lumimask', 'met_filters', 'met_triggers', 'exclude_wjets_greater_120', 'exclude_wjets_less_120', 'QCD_NoGenIsoPho', 'recoil_sr', 'mindphi_sr', 'minDphi_sr', 'GJet_GenIsoPho', 'jetveto', 'one_ak15', 'leading_fj250', 'iszeroL', 'noextrab', 'recoil_wmcr', 'mindphi_wmcr', 'minDphi_wmcr', 'isoneM', 'met150', 'single_electron_triggers', 'recoil_wecr', 'mindphi_wecr', 'minDphi_wecr', 'isoneE', 'recoil_tmcr', 'mindphi_tmcr', 'minDphi_tmcr', 'extrab', 'recoil_tecr', 'mindphi_tecr', 'minDphi_tecr', 'recoil_zmcr', 'mindphi_zmcr', 'minDphi_zmcr', 'istwoM', 'met120', 'dimu_mass', 'recoil_zecr', 'mindphi_zecr', 'minDphi_zecr', 'istwoE', 'diele_mass', 'single_photon_triggers', 'recoil_gcr', 'mindphi_gcr', 'minDphi_gcr', 'isoneG'], growth=True, name='cutname')